# CIFAR-10 CNN with Batch Normalization and Dropout

**Question:** Use the CIFAR dataset to build a deep neural network model and analyze the effect of batch normalization and dropout.

---

## What is CIFAR-10?
CIFAR-10 is a dataset of 60,000 small (32×32 pixel) color images across 10 classes:
airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

## Key Techniques Used

### Batch Normalization
After each convolutional layer, we normalize the outputs so they have a mean of 0 and variance of 1.
- **Why?** Without it, values can grow very large or small as they pass through many layers, making training unstable and slow.
- **Effect:** Speeds up training, allows higher learning rates, reduces sensitivity to weight initialization.

### Dropout
During training, randomly "turns off" a fraction of neurons (we use 50%).
- **Why?** Prevents the network from relying too heavily on any single neuron (overfitting).
- **Effect:** Acts as regularization — the model generalizes better to unseen data. Disabled automatically during evaluation.

In [ ]:
# Core PyTorch libraries
import torch
import torch.nn as nn                        # Neural network building blocks
import torch.optim as optim                  # Optimizers (Adam, SGD, etc.)
import torchvision                           # Datasets and pretrained models
import torchvision.transforms as transforms  # Image preprocessing tools
from torch.utils.data import DataLoader, Subset

## Step 1 — Load the Data

We convert images to PyTorch tensors (arrays of numbers the model can process).
We use a small subset (200 training, 50 test images) so training finishes quickly.

In [ ]:
# ToTensor converts PIL images (H x W x C uint8) → PyTorch tensors (C x H x W float32 in [0,1])
transform = transforms.ToTensor()

# Download CIFAR-10; train=True for training split, train=False for test split
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Use only a small subset so training is fast (remove Subset() to use the full dataset)
train_subset = Subset(train_dataset, range(200))
test_subset  = Subset(test_dataset,  range(50))

# DataLoader feeds data in batches to the model
# shuffle=True mixes training data each epoch so the model doesn't memorize order
train_loader = DataLoader(train_subset, batch_size=10, shuffle=True)
test_loader  = DataLoader(test_subset,  batch_size=10, shuffle=False)

## Step 2 — Define the Model

We use a **Convolutional Neural Network (CNN)**. CNNs are great for images because they learn spatial patterns (edges, textures, shapes) rather than treating each pixel independently.

### Architecture
```
Input image (3 × 32 × 32)
  → Conv Block 1:  Conv → BatchNorm → ReLU → MaxPool   →  32 × 16 × 16
  → Conv Block 2:  Conv → BatchNorm → ReLU → MaxPool   →  64 ×  8 ×  8
  → Flatten                                            →  4096 values
  → Fully Connected 1: 4096 → 512 + ReLU + Dropout
  → Fully Connected 2: 512  → 10  (one score per class)
```

- **Conv2d**: Learns to detect features (edges, shapes) in the image
- **BatchNorm2d**: Normalizes conv output → stable, faster training
- **ReLU**: Activation function — replaces negative values with 0, adds non-linearity
- **MaxPool2d(2)**: Halves spatial size by keeping only the max value in each 2×2 region
- **Flatten**: Turns the 3D feature map into a 1D vector for the dense layers
- **Dropout(0.5)**: Randomly zeros 50% of neurons during training to prevent overfitting

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # --- Convolutional Block 1 ---
        # Input:  3 channels  (RGB), 32×32
        # Output: 32 channels (learned features), 16×16  (halved by MaxPool)
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),  # Normalize across the 32 feature maps
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)  # 32×32 → 16×16
        )

        # --- Convolutional Block 2 ---
        # Input:  32 channels, 16×16
        # Output: 64 channels (richer features), 8×8  (halved by MaxPool)
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)  # 16×16 → 8×8
        )

        # --- Fully Connected (Dense) Layers ---
        self.flatten = nn.Flatten()           # 64 × 8 × 8 = 4096 values → 1D vector
        self.fc1     = nn.Linear(4096, 512)   # Compress 4096 features down to 512
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5)      # Drop 50% of neurons randomly during training
        self.fc2     = nn.Linear(512, 10)     # Output 10 scores, one per CIFAR-10 class

    def forward(self, x):
        """Defines how data flows through the network."""
        x = self.conv_block1(x)  # Extract low-level features
        x = self.conv_block2(x)  # Extract higher-level features
        x = self.flatten(x)      # Flatten to 1D
        x = self.fc1(x)          # Dense layer
        x = self.relu(x)
        x = self.dropout(x)      # Regularize (only active during training)
        x = self.fc2(x)          # Final class scores (logits)
        return x

## Step 3 — Train and Evaluate

### Training loop (one epoch)
1. Feed a batch of images → get predictions
2. Calculate **loss** (how wrong the predictions are) using Cross-Entropy
3. **Backpropagate**: compute how much each weight contributed to the error
4. **Optimizer step**: nudge weights in the direction that reduces loss

### Evaluation loop
- Run with `torch.no_grad()` — no gradient tracking needed, saves memory
- Dropout is automatically disabled in eval mode

In [ ]:
def train_and_evaluate(model, optimizer, criterion, num_epochs, device):
    model.to(device)  # Move model to CPU or GPU

    for epoch in range(num_epochs):

        # ── Training Phase ──────────────────────────────────────────
        model.train()  # Enable training mode (activates Dropout, BatchNorm tracks stats)
        train_loss    = 0.0
        train_correct = 0
        train_total   = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()          # Clear gradients from previous batch
            outputs = model(images)        # Forward pass: get predictions
            loss = criterion(outputs, labels)  # Compare predictions to true labels
            loss.backward()                # Backward pass: compute gradients
            optimizer.step()               # Update weights using gradients

            train_loss += loss.item()
            predicted   = torch.argmax(outputs, dim=1)  # Pick class with highest score
            train_total   += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # ── Evaluation Phase ─────────────────────────────────────────
        model.eval()  # Disable Dropout; BatchNorm uses stored stats
        test_loss    = 0.0
        test_correct = 0
        test_total   = 0

        with torch.no_grad():  # No gradient computation needed during evaluation
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss    = criterion(outputs, labels)

                test_loss    += loss.item()
                predicted     = torch.argmax(outputs, dim=1)
                test_total   += labels.size(0)
                test_correct += (predicted == labels).sum().item()

        # ── Print Summary ────────────────────────────────────────────
        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = (train_correct / train_total) * 100
        avg_test_loss  = test_loss  / len(test_loader)
        test_accuracy  = (test_correct  / test_total)  * 100

        print(f"Epoch {epoch+1:2d}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.1f}% | "
              f"Test  Loss: {avg_test_loss:.4f},  Test  Acc: {test_accuracy:.1f}%")


# ── Setup and Run ───────────────────────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available
model     = CNN()
criterion = nn.CrossEntropyLoss()  # Standard loss for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam adapts learning rate automatically

print(f"Training on: {device}")
train_and_evaluate(model, optimizer, criterion, num_epochs=30, device=device)